# Module 11 - 실습 2 솔루션: Self-Reflection 루프

In [ ]:
import sys
sys.path.insert(0, '../..')

import json
from typing import TypedDict
from langgraph.graph import StateGraph, END
from common.fake_llm import FakeLLM

print("Self-Reflection 솔루션 준비 완료")

MAX_REFLECTION_ROUNDS = 2

In [ ]:
class ReflectionState(TypedDict):
    topic: str
    draft: str
    reflection: dict | None
    reflection_count: int
    final_output: str | None

In [ ]:
generate_llm = FakeLLM(responses={
    "피드백": "피드백을 반영하여 수정된 내용입니다. asyncio는 비동기 프로그래밍의 핵심입니다.",
    "default": "asyncio는 Python의 비동기 프로그래밍 라이브러리입니다. async/await 키워드를 사용합니다.",
})

reflect_responses = [
    '{"approved": false, "score": 6, "feedback": "더 구체적인 예시가 필요합니다."}',
    '{"approved": true, "score": 9, "feedback": ""}',
]
reflect_call_count = [0]

def get_reflect_response():
    idx = min(reflect_call_count[0], len(reflect_responses) - 1)
    reflect_call_count[0] += 1
    return reflect_responses[idx]

In [ ]:
def generate_node(state: ReflectionState) -> dict:
    """초안을 생성하거나 피드백을 반영하여 수정합니다."""
    reflection = state.get("reflection")
    
    if reflection and not reflection.get("approved", True):
        feedback = reflection.get("feedback", "")
        prompt = f"피드백을 반영하여 수정해주세요: {feedback}"
    else:
        prompt = f"다음 주제로 문서를 작성해주세요: {state['topic']}"
    
    draft = generate_llm.invoke(prompt).content
    return {"draft": draft}

def reflect_node(state: ReflectionState) -> dict:
    """초안을 검토합니다."""
    reflection_json = get_reflect_response()
    result = json.loads(reflection_json)
    
    return {
        "reflection": result,
        "reflection_count": state.get("reflection_count", 0) + 1,
    }

def decide_after_reflection(state: ReflectionState) -> str:
    """검토 결과에 따라 다음 경로를 결정합니다."""
    reflection = state.get("reflection", {})
    count = state.get("reflection_count", 0)
    
    if reflection.get("approved", True) or count >= MAX_REFLECTION_ROUNDS:
        return "finalize"
    
    return "generate"

def finalize_node(state: ReflectionState) -> dict:
    """최종 결과를 확정합니다."""
    count = state.get("reflection_count", 0)
    score = state.get("reflection", {}).get("score", "N/A")
    
    return {
        "final_output": (
            f"[검토 {count}회 완료, 점수: {score}]\n\n"
            f"{state['draft']}"
        ),
    }

In [ ]:
def build_reflection_graph():
    """Self-Reflection 루프 그래프를 구성합니다."""
    graph = StateGraph(ReflectionState)
    
    graph.add_node("generate", generate_node)
    graph.add_node("reflect", reflect_node)
    graph.add_node("finalize", finalize_node)
    
    graph.set_entry_point("generate")
    graph.add_edge("generate", "reflect")
    
    # 순환 엣지: reflect → generate (미승인) 또는 finalize (승인)
    graph.add_conditional_edges(
        "reflect",
        decide_after_reflection,
        {
            "generate": "generate",
            "finalize": "finalize",
        },
    )
    graph.add_edge("finalize", END)
    
    return graph.compile()

app = build_reflection_graph()
print("그래프 구성 완료")

In [ ]:
reflect_call_count[0] = 0

result = app.invoke({
    "topic": "Python asyncio",
    "draft": "",
    "reflection": None,
    "reflection_count": 0,
    "final_output": None,
})

print(f"검토 횟수: {result['reflection_count']}")
print(f"최종 결과:\n{result['final_output']}")

In [ ]:
# 검증
assert result["final_output"] is not None
assert result["reflection_count"] > 0
assert result["reflection_count"] <= MAX_REFLECTION_ROUNDS + 1
assert "검토" in result["final_output"]
print(f"✅ 실습 3 완료! Self-Reflection 루프가 {result['reflection_count']}회 실행되었습니다.")